# 第8回 教師あり解析パイプライン演習


---


## 前回からの接続


第1回から第7回までで、教師あり学習の道具がそろいました。データを見る、前処理をする、モデルを立てる、交差検証（cross-validation）で測る、テストで確かめる、係数を読む、しきい値を決める。

今日は、新しいデータで、この道具を最初から最後まで一周します。新しい書き方はほとんど出てきません。そのかわり、最後に、今日のコードを関数に分けた1本のスクリプト（上から順に実行すれば最後まで動く、ひとまとまりのコード）にまとめます。


---


## 今日の分析目標


あなたは南極の調査担当者です。くちばしや翼の長さ、体重といった測定値から、ペンギンの種（3種）を見分けたい。そして、どの測定が見分けに役立つかを、調査チームに報告したい。

データの理解から報告までを順に進めます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは南極の調査チームで、データをまとめる担当者です。チームは島をまわってペンギンを測り、くちばしの長さと深さ、翼の長さ、体重を記録します。どの島にいたか、オスかメスかも書きとめます。

決めたいのは、この測定値からペンギンの種（3種）を見分けられるかどうかです。そして、どの測定が見分けに役立つかを、調査チームに報告したいところです。


### 手元のデータ

南極のパーマー基地の近くで測られた、344羽のペンギンの記録です（palmerpenguins。Kristen Gorman博士とPalmer Station Antarctica LTERが集めたもの）。1行が1羽です。

答えの列`species`は種で、Adelie・Chinstrap・Gentooの3種です。


### 最後に出す答え

3種をテストで何羽中何羽当てられるかと、モデルが見分けに主に使った測定はどれかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 新しいデータで、欠けた値と区分の列を確かめ、前処理を設計できる
- パイプライン（pipeline）を交差検証で測り、Cを選び、テストで一度だけ確かめられる
- 3種を見分ける混同行列（confusion matrix）と係数を読み、どの測定が役立つかを説明できる
- 読み込み・組み立て・評価・報告を関数に分け、1本のスクリプトにまとめられる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は総合演習なので、①から③までを全部回ります。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (1, 2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. データを理解する


見分ける前に、まず種ごとの違いと、欠けた値を見ます。南極のパーマー基地の近くで測られた、344羽のペンギンのデータです（palmerpenguins）。1行が1羽で、`species`が種（Adelie・Chinstrap・Gentooの3種）です。


### お手本：読み込んで、種ごとに比べる

- 役割：データを読み込み、種ごとの羽数と、測定値の平均を比べる
- 入力：データの置き場所とファイル名
- 出力：表`df`、種ごとの羽数と平均


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/penguins.csv')
print(df['species'].value_counts())
num = ['bill_length_mm', 'bill_depth_mm',
       'flipper_length_mm', 'body_mass_g']
print(df.groupby('species')[num].mean().round(1))


Adelieが152羽、Gentooが124羽、Chinstrapが68羽。測定値は、くちばしの長さ（`bill_length_mm`）と深さ（`bill_depth_mm`）、翼の長さ（`flipper_length_mm`）、体重（`body_mass_g`）の4本です。

平均を並べると、Adelieはくちばしが短く（38.8 mm）、Gentooはくちばしが浅く（15.0 mm）、翼が長く体が重い（217 mm・5076 g）。Chinstrapはくちばしが長く（48.8 mm）、深さ・翼・体重はAdelieとほぼ同じです。


### 変えてみる：種と島を、クロス集計する

区分の列は、島（`island`）と性別（`sex`）の2本です。`pd.crosstab`は、2つの区分の組み合わせごとに数を数えた表を作ります。


In [ ]:
print(pd.crosstab(df['species'], df['island']))


ChinstrapはDream島にしかいません。GentooはBiscoe島にしかいません。Adelieは3つの島すべてにいます。島は、種を見分ける強い手がかりになりそうです。


### TODO①：欠けた値を数える

列ごとに欠けた値がいくつあるかを数え、全体の合計も表示してください。


In [ ]:
# TODO: 列ごとの欠けた値の数と、その合計を表示する
# ヒント: df.isna().sum() で列ごと、さらに .sum() で合計
...


測定値の4本はそれぞれ2つ、性別は11羽が欠けています。合計19か所です。区分の列（性別）にも欠けがあるので、区分の列にも補う手順が要ります。


---


## 2. 前処理を設計する


1節でわかったことを、前処理の設計に移します。

- 測定値（数の列）：欠けがある → 中央値で補う。単位がばらばら（mmとg）→ 標準化（standardization）
- 島と性別（区分の列）：性別に欠けがある → 最頻値で補う。区分 → ワンホット（one-hot）


### おさらい：Pipeline

```python
Pipeline([('名前1', 部品1), ('名前2', 部品2)])
```

部品を順に並べて、1本の部品にまとめる書き方でした（第7回）。`fit`を呼ぶと、前から順に訓練で学びます。


### お手本：分けて、前処理を組む

- 役割：訓練とテストに分け、数の列と区分の列に別々の手順を当てる前処理を組む
- 入力：表`df`
- 出力：`X_tr`・`X_te`・`y_tr`・`y_te`と、前処理の部品`pre`


In [ ]:
cat = ['island', 'sex']
X, y = df[num + cat], df['species']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42, stratify=y)
num_steps = Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())])
cat_steps = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('oh', OneHotEncoder(handle_unknown='ignore'))])
pre = ColumnTransformer([('num', num_steps, num),
                         ('cat', cat_steps, cat)])
print(len(y_tr), len(y_te))


訓練275羽、テスト69羽。`stratify=y`で、3種の割合をそろえて分けました。


### TODO②：前処理のあとの列を確かめる

`pre`を訓練`X_tr`で`fit`し、`get_feature_names_out()`で、前処理のあとの列の名前を表示してください。何本になりましたか。


In [ ]:
# TODO: pre を訓練で fit し、前処理のあとの列の名前を表示する
# ヒント: names = pre.fit(X_tr).get_feature_names_out()
#         print(names, len(names))
...


9本です。測定値の4本（`num__`で始まる）と、島の3本（Biscoe・Dream・Torgersen）、性別の2本（Female・Male）。区分の列が、ワンホットで5本の列に分かれました。


---


## 3. パイプラインと交差検証


前処理`pre`で列は9本になったので、次はモデルとつないで、テストを見ずに訓練の中で実力を測ります。

### お手本：モデルまで1本にして、交差検証で測る

- 役割：前処理とロジスティック回帰（logistic regression）をパイプラインにまとめ、訓練の中の5分割交差検証で正解率を測る
- 入力：`pre`、訓練データ
- 出力：5回ぶんの正解率と、その平均

3種の分類（classification）でも、ロジスティック回帰はそのまま使えます。種ごとに「その種である確率」を出し、一番高い種を答えます。


In [ ]:
pipe = Pipeline([('pre', pre),
                 ('model', LogisticRegression(max_iter=5000))])
cv = cross_val_score(pipe, X_tr, y_tr, cv=5)
print(cv.round(3), round(cv.mean(), 3))


5回の正解率は0.982〜1.000、平均0.993。テストを見る前に、訓練の中だけで、とても高い実力が見えました。


### 変えてみる：GridSearchCVでCを選ぶ

第7回と同じく、Cの候補を辞書で渡します。


In [ ]:
grid = {'model__C': [0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(pipe, grid, cv=5).fit(X_tr, y_tr)
print(gs.best_params_, round(gs.best_score_, 3))


選ばれたのはC＝10、交差検証の正解率は0.996です。


### TODO③：区分の列は、役に立っているか

区分の列（島・性別）を外して、測定値の4本だけのパイプラインを作り、同じように交差検証の正解率の平均を出してください。お手本（0.993）と比べて、区分の列は役に立っていますか。


In [ ]:
# TODO: 数の列だけのパイプラインで、交差検証の正解率の平均を出す
# ヒント: pre_n = ColumnTransformer([('num', num_steps, num)])
#         pipe_n = Pipeline([('pre', pre_n), ('model', …)])
#         （model はお手本と同じ）
...


測定値だけでは、平均0.975。区分の列を入れた0.993より下がります。区分の列（島・性別）は役に立っていました。

ただし、ここでは島と性別を両方外したので、どちらが役に立ったのかは、この比べ方だけでは分けられません。1節のクロス集計（ChinstrapはDream島だけ、GentooはBiscoe島だけ）から、島が役に立った見込みはあります。でも、まだ確かめてはいません。確かめるなら、島だけ・性別だけを測定値に足して、交差検証で比べます。


---


## 4. テストと混同行列


3節では、交差検証でC＝10のパイプライン`gs`を選んだので、次はテストで確かめます。

### お手本：テストで一度だけ確かめる

- 役割：選んだパイプライン`gs`で、テストの正解率と、3種の混同行列を出す
- 入力：`gs`、テストデータ
- 出力：正解率と、3行3列の混同行列

テストを使うのは、ここで一度だけです。テストは、最後まで取っておく本番でした（第2回）。何度もテストを見てモデルを直すと、テストに合わせこんでしまいます。7節でもう一度テストの数を出しますが、それは同じ計算を関数で書き直して、同じ数が出るかを見るだけです。モデルを選び直すことはしません。


In [ ]:
labels = ['Adelie', 'Chinstrap', 'Gentoo']
print(round(gs.score(X_te, y_te), 3))
print(confusion_matrix(y_te, gs.predict(X_te), labels=labels))


テストの正解率は0.986。69羽のうち68羽を当てました。混同行列は、行が本当の種、列が判定です。外した1羽は、本当はAdelieなのにChinstrapと判定されました。Gentooの25羽は、すべて正解です。


In [ ]:
# @title 図：3種の混同行列
cm = confusion_matrix(y_te, gs.predict(X_te), labels=labels)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, v, ha='center', va='center', fontsize=16,
            color='white' if v > cm.max() / 2 else '#1a1a2e')
ax.set_xticks(range(3), labels); ax.set_yticks(range(3), labels)
ax.set_xlabel('判定'); ax.set_ylabel('本当'); plt.tight_layout(); plt.show()


---


## 5. どの測定が役立ったか：係数と散布図


3種は見分けられたので、残る問いは、どの測定が見分けに役立つかです。3種の分類のロジスティック回帰は、種ごとに1組の係数を持ちます。Adelieの係数は「Adelieらしさ」を、Gentooの係数は「Gentooらしさ」を決めます。測定値の4本は標準化されているので、その4本どうしなら、係数の大きさ（絶対値）で比べられます（第4回）。島と性別のワンホットの列は0/1のままで、標準化されていません。単位が違うので、測定値の係数と大きさをそのまま比べることはできません。だから、ここでは測定値の4本だけを大きい順に並べます。


### お手本：Adelieらしさを決める列

- 役割：選んだモデルから、Adelieの係数を取り出し、測定値の4本を絶対値の大きい順に表示する
- 入力：`gs.best_estimator_`
- 出力：列の名前と係数

`named_steps`は、パイプラインの部品を名前で取り出す辞書です。`coef_[0]`は1行目、つまり1番目の種（Adelie）の係数です。`coef[:4]`は先頭の4本で、測定値（`num__`の列）にあたります。`np.argsort`は小さい順の並び順を返すので、`[::-1]`で逆にして大きい順にします。


In [ ]:
best = gs.best_estimator_
names = best.named_steps['pre'].get_feature_names_out()
coef = best.named_steps['model'].coef_[0]       # Adelie らしさ
top = np.argsort(np.abs(coef[:4]))[::-1]       # 測定値の4本
for j in top:
    print(f'{names[j]:24s} {coef[j]:+.2f}')


絶対値がいちばん大きいのはくちばしの長さ（−4.43）。マイナスなので、「くちばしが短いほどAdelieらしい」です。次がくちばしの深さ（＋1.88）です。1節の平均（Adelieはくちばしが短い）と合っています。


### TODO④：Gentooらしさを決める列

お手本をまねて、Gentoo（3番目の種、`coef_[2]`）の測定値の係数を、絶対値の大きい順に表示してください。


In [ ]:
# TODO: Gentoo の測定値の係数を、絶対値の大きい順に表示する
# ヒント: お手本の coef_[0] を coef_[2] に変える
...


Gentooらしさで絶対値がいちばん大きいのはくちばしの深さ（−2.15）。マイナスなので、「くちばしが浅いほどGentooらしい」です。1節の平均（Gentooはくちばしが浅い）と合っています。


In [ ]:
# @title 図：くちばしの長さと深さ
fig, ax = plt.subplots(figsize=(7, 4.6))
for sp, c in zip(labels, ['#0066cc', '#2a9d8f', '#e63946']):
    d = df[df['species'] == sp]
    ax.scatter(d['bill_length_mm'], d['bill_depth_mm'], s=18, alpha=0.7,
               color=c, label=sp)
ax.set_xlabel('くちばしの長さ (mm)'); ax.set_ylabel('くちばしの深さ (mm)')
ax.legend(); plt.tight_layout(); plt.show()


散布図で、係数の話を目で確かめます。Adelie（青）は左（くちばしが短い）に、Gentoo（赤）は下（くちばしが浅い）に固まっています。Chinstrap（緑）は右上（くちばしが長くて深い）です。くちばしの長さと深さの2本だけで、3種がかなりきれいに分かれます。


---


## 6. 答えを出す：調査チームへの報告


4節で見分ける力を、5節でモデルが使った測定を確かめました。最初の問いに戻ります。調査担当者として、ここまでの結果を報告にまとめます。

答えは「くちばし・翼・体重の4つの測定と、島・性別から、3種を見分けられる。交差検証の正解率は0.996、テストでは69羽のうち68羽（0.986）を当てた。このモデルが主に使ったのは、Adelieではくちばしの長さ、Gentooではくちばしの深さ。外した1羽はAdelieをChinstrapと取り違えたもの」。

注意：係数の順位は「このモデルが主に使った列」であって、測定そのものが見分けに役立つ順とは限りません。似た動きをする列どうし（たとえば翼の長さと体重）は、係数を分け合うからです。

まだ言えないことも、2つ残っています。

- くちばしを測れなかったら：モデルが主に使った測定がなくても、見分けられるか（課題）
- 新しい1羽：学んだモデルを保存して、あとで新しく測った1羽を判定できるか（発展）


---


## 7. 1本のスクリプトにまとめる


ここまでのコードは、ノートのあちこちに散らばっています。別のデータや、別の列の組み合わせでやり直したいとき、どのセルをどの順で実行し直せばよいか、迷いそうです。

そこで、今日の手順を4つの関数に分けて、1つのセルにまとめます。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。


### お手本：4つの関数と、それを呼ぶ3行

- 役割：読み込み（`load`）・組み立て（`build_pipeline`）・交差検証（`evaluate`）・最後のテスト（`final_test`）の4つの関数を作り、交差検証まで順に呼ぶ
- 入力：ファイルの場所、列の並び、C
- 出力：交差検証の正解率

上の関数の定義だけを読めば、「何をする部品があるか」がわかります。下の3行だけを読めば、「どの順で何をしたか」がわかります。

`evaluate`（交差検証）と`final_test`（テスト）を、別の関数に分けたのがポイントです。比べたり選んだりするのは`evaluate`の値だけにして、`final_test`は、選び終えたあとに一度だけ呼びます。


In [ ]:
def load(path):                               # 読み込んで、分ける
    df = pd.read_csv(path)
    X, y = df.drop(columns='species'), df['species']
    return train_test_split(X, y, test_size=0.2,
                            random_state=42, stratify=y)

def build_pipeline(num_cols, cat_cols, C=1.0):   # 前処理＋モデル
    num_steps = Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('sc', StandardScaler())])
    cat_steps = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore'))])
    pre = ColumnTransformer([('num', num_steps, num_cols),
                             ('cat', cat_steps, cat_cols)])
    model = LogisticRegression(C=C, max_iter=5000)
    return Pipeline([('pre', pre), ('model', model)])

def evaluate(model, X_tr, y_tr):                 # 訓練の中で交差検証
    return cross_val_score(model, X_tr, y_tr, cv=5).mean()

def final_test(model, X_tr, y_tr, X_te, y_te):   # 一度だけテスト
    model.fit(X_tr, y_tr)
    return model.score(X_te, y_te)

X_tr, X_te, y_tr, y_te = load(f'{DATA_DIR}/penguins.csv')
model = build_pipeline(num, cat, C=10)
print('交差検証', round(evaluate(model, X_tr, y_tr), 3))


交差検証は0.996。3節のGridSearchCV（C＝10）と同じ数が、3行で出ました。`load`の中の`df`は、関数の中だけの名前です。外の`df`を書きかえることはありません。


### 変えてみる：Cを変えて、交差検証で比べる

関数に分けてあれば、やり直しは呼ぶ行を変えるだけです。Cを0.01（強く縮める）にして、交差検証で比べます。テストは見ません。


In [ ]:
weak = build_pipeline(num, cat, C=0.01)
print('交差検証', round(evaluate(weak, X_tr, y_tr), 3))


交差検証は0.924と下がりました。縮めすぎです。3節のGridSearchCVが0.01を選ばなかったことと合っています。


### お手本：選び終えたら、最後に一度だけテスト

選んだC＝10のパイプライン`model`を、訓練全体で学び、テストで一度だけ確かめます。


In [ ]:
print('テスト', round(final_test(model, X_tr, y_tr, X_te, y_te), 3))


テストは0.986。4節と同じ数です。4節の計算を関数で書き直しただけで、モデルを選び直してはいません。

このように、定義（部品）と呼び出し（手順）を分けて書くと、条件を変えたやり直しが楽になり、読む人にも流れが見えます。課題では、この関数を使って、くちばしの長さを外した場合を試します。


---


## 目標に答えられたか


- 今日の目標は「測定値からペンギンの3種を見分け、どの測定が役立つかを報告する」ことでした。6節の答えで、このモデルが主に使った測定は何でしたか？
- 1節のクロス集計から、島についてどんなことがわかりましたか？ それはTODO③の結果とどうつながりましたか？
- 4節で、テストを「一度だけ」使うのはなぜでしたか？
- 7節の4つの関数は、それぞれ何をする部品でしたか？


---


## 今日の要点


- 総合演習：データの理解 → 前処理の設計 → パイプラインと交差検証 → テスト → 係数と散布図 → 報告
- 前処理は、1節で見たこと（欠けた値、単位、区分）から設計する
- 3種の分類でも、ロジスティック回帰は種ごとに係数を持つ。係数の符号と大きさを、平均や散布図と照らし合わせる。大きさを比べるのは、標準化した測定値どうし
- テストは最後に一度だけ使う。比べたり選んだりするのは、交差検証の値で
- Python：手順を関数に分け、定義（部品）と呼び出し（手順）を分けて1本のスクリプトにまとめる


---


## 次回へ


ここまでは、いつも正解のラベル（利用台数、価格、心臓病、種）がありました。

次回からは、ラベルのないデータに移ります。まずは、たくさんの列を持つデータを、少ない本数に縮めます。

次回の問いは「400枚の顔を、見分ける力を落とさずに持つには、1枚あたり何個の数でよいか？」です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

野外では、くちばしの長さを測れないことがあるとします。7節の関数を使って、くちばしの長さ（`bill_length_mm`）を外した3本の測定値と区分の列で、C＝10のパイプラインを組み、交差検証とテストの正解率を報告してください。さらに、テストの混同行列を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`num3 = ['bill_depth_mm', 'flipper_length_mm', 'body_mass_g']`を作り、
`build_pipeline(num3, cat, C=10)` → `evaluate(…)`で交差検証、最後に`final_test(…)`でテストを一度だけ呼びます。
混同行列は、4節と同じく`confusion_matrix(y_te, model3.predict(X_te), labels=labels)`です。

</details>


### 応用②（判断）

応用①の、テストの正解率はいくつですか。小数第2位まで答えてください（例: 0.57）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`final_test`が返す値が、テストの正解率です。`round(値, 2)`で丸めます。

</details>


### 応用③（解釈）

応用①の混同行列で、どの種とどの種の取り違えが増えましたか。5節の散布図を手がかりに、その理由と、現場の測定への提案を、調査チームに向けて3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### 学んだモデルを保存して、新しい1羽を判定する

調査の現場では、学習は一度だけにして、あとは新しく測った個体を判定したいということがよくあります。`joblib`を使うと、前処理からモデルまでのパイプラインを丸ごとファイルに保存し、あとで読み込んで使えます。


In [ ]:
import os
import joblib

model = build_pipeline(num, cat, C=10).fit(X_tr, y_tr)
joblib.dump(model, 'penguin_model.joblib')          # 保存する
print(os.path.getsize('penguin_model.joblib'), 'バイト')
loaded = joblib.load('penguin_model.joblib')        # 読み込む
print([name for name, _ in loaded.steps])


In [ ]:
# 新しく測った3羽（2羽目は性別が未記入）
new_birds = pd.DataFrame(
    [[45.0, 18.5, 195.0, 3800.0, 'Dream', 'Male'],
     [39.0, 18.5, 190.0, 3700.0, 'Torgersen', np.nan],
     [47.0, 15.0, 215.0, 5000.0, 'Biscoe', 'Female']],
    columns=num + cat)
proba = pd.DataFrame(loaded.predict_proba(new_birds),
                     columns=loaded.classes_).round(3)
proba['判定'] = loaded.predict(new_birds)
print(proba)
os.remove('penguin_model.joblib')                  # 後片付け


読み方　保存したファイルは約5 KBです。前処理からモデルまでの一式が、この小さなファイルに収まっています。

1羽目（くちばし45.0 mm・Dream島）は、判定はChinstrapですが、確率は0.633（Adelie 0.359）と6割ほどです。くちばしの長さ45 mmは、Adelie（平均38.8 mm）とChinstrap（平均48.8 mm）の間の、ややChinstrap寄りです。`predict`の答えだけでなく確率も見て、この個体は「要確認」として扱うのが安全です。

2羽目は、性別を`np.nan`（欠けた値）にしてあります。それでも止まりません。保存された補う部品が訓練の最頻値（Male）で埋めてから、Adelie（1.000）と判定します。

`np.nan`でなく`None`と書くと、`SimpleImputer`は欠けた値と見なさず、補いません。そのままワンホットに渡り、知らない区分として性別の列がすべて0になります（`handle_unknown='ignore'`なので止まりはしません）。欠けた値は`np.nan`で書きましょう。3羽目はGentoo（0.999）です。

注意：保存したファイルは、保存したときのscikit-learnの版と結びついています。別の版で読み込むと、警告が出ることがあります。

試すなら、1羽目の`bill_length_mm`を42や48に変えて、AdelieとChinstrapの確率がどこで入れかわるかを見てみましょう。
